# ⚡ CloudEdge — GRPO Training Notebook

This notebook trains a **Qwen2.5-0.5B-Instruct** model using **Group Relative Policy Optimization (GRPO)** to control a multi-agent cloud crisis simulator.

Built for the **Meta PyTorch OpenEnv Hackathon Grand Finale**.

**Runtime:** Select **GPU → T4** before running.

## 1. Install Dependencies

In [ ]:
!pip install -q trl transformers datasets accelerate peft bitsandbytes

## 2. Configuration

In [ ]:
import os, random, re
import numpy as np
import torch
from datetime import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "outputs/ecocloud-grpo"
TRAIN_PROMPTS = 64
TRAIN_STEPS = 512

ACTIONS = ["scale_up", "scale_down", "optimize_energy", "migrate_region"]
EFFECTS = {
    "scale_up":        {"latency": -40, "cost": +30, "carbon": +20},
    "scale_down":      {"latency": +25, "cost": -35, "carbon": -15},
    "optimize_energy": {"latency": +10, "cost": -20, "carbon": -40},
    "migrate_region":  {"latency": +15, "cost": +10, "carbon": -50},
}
TARGETS = {"latency": 150, "cost": 400, "carbon": 220}

print(f"Model: {MODEL_NAME}")
print(f"Steps: {TRAIN_STEPS}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 3. System Prompt & Dataset

In [ ]:
SYSTEM_PROMPT = (
    "You are the CloudEdge controller managing a cloud platform in crisis.\n"
    "Pick the BEST single action for the current state. Respond with ONLY the action name.\n\n"
    "Actions:\n"
    "  scale_up        → latency -40, cost +30, carbon +20\n"
    "  scale_down      → latency +25, cost -35, carbon -15\n"
    "  optimize_energy → latency +10, cost -20, carbon -40\n"
    "  migrate_region  → latency +15, cost +10, carbon -50\n\n"
    "Targets: latency<150ms, cost<$400, carbon<220"
)

def make_prompt(latency, cost, carbon, load="critical"):
    user_msg = f"Cloud state: latency={latency}ms, cost=${cost}/hr, carbon={carbon}, load={load}. Best action?"
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]

# Generate varied training prompts
prompts = []
for i in range(TRAIN_PROMPTS):
    lat = random.randint(180, 350)
    cst = random.randint(420, 700)
    crb = random.randint(230, 500)
    prompts.append({"prompt": make_prompt(lat, cst, crb)})

dataset = Dataset.from_list(prompts)
print(f"Created {len(dataset)} training prompts")
print(f"Example: {prompts[0]['prompt'][1]['content']}")

## 4. Shaped Reward Function

The reward evaluates each action's impact on latency, cost, and carbon:

```
reward = Σ (gap_closure × weight) + worst_metric_bonus
```

In [ ]:
def compute_shaped_reward(action_name, state):
    effects = EFFECTS.get(action_name)
    if effects is None:
        return -5.0
    reward = 0.0
    for metric in ["latency", "cost", "carbon"]:
        current = state[metric]
        target = TARGETS[metric]
        delta = effects[metric]
        if current > target:
            gap = current - target
            if delta < 0:
                reward += min(abs(delta), gap) * 0.1
            else:
                reward -= delta * 0.05
        else:
            if delta > 0:
                reward -= delta * 0.15
    gaps = {m: state[m] - TARGETS[m] for m in ["latency", "cost", "carbon"]}
    worst = max(gaps, key=gaps.get)
    if effects[worst] < 0:
        reward += 2.0
    return round(reward, 2)

# Show scoring example
crisis = {"latency": 280, "cost": 620, "carbon": 380}
print("Scoring example (crisis state):")
for a in ACTIONS:
    print(f"  {a:20s} → reward: {compute_shaped_reward(a, crisis)}")

## 5. Reward Function for GRPO

In [ ]:
def reward_func(completions, prompts=None, **kwargs):
    rewards = []
    for idx, completion in enumerate(completions):
        if isinstance(completion, list):
            text = completion[-1]["content"] if completion else ""
        else:
            text = str(completion)
        text_lower = text.strip().lower()
        chosen = None
        for a in ACTIONS:
            if a in text_lower:
                chosen = a
                break
        # Index-based state variation for non-zero advantages
        base_lat, base_cst, base_crb = 280, 620, 380
        noise_lat = (idx * 7) % 30 - 15
        noise_cst = (idx * 11) % 40 - 20
        noise_crb = (idx * 13) % 30 - 15
        state = {
            "latency": base_lat + noise_lat + random.gauss(0, 5),
            "cost": base_cst + noise_cst + random.gauss(0, 8),
            "carbon": base_crb + noise_crb + random.gauss(0, 5),
        }
        if chosen:
            r = compute_shaped_reward(chosen, state)
        else:
            r = -5.0
        rewards.append(r)
    return rewards

print("Reward function ready.")

## 6. Train with GRPO

In [ ]:
config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    max_steps=TRAIN_STEPS,
    per_device_train_batch_size=2,
    num_generations=4,
    max_completion_length=16,
    max_prompt_length=300,
    logging_steps=32,
    save_steps=128,
    learning_rate=5e-6,
    report_to="none",
)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

trainer = GRPOTrainer(
    model=model,
    config=config,
    reward_funcs=reward_func,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print(f"Starting GRPO training — {TRAIN_STEPS} steps...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved to {OUTPUT_DIR}")

## 7. Test the Trained Model

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model=OUTPUT_DIR, tokenizer=OUTPUT_DIR, max_new_tokens=16)

test_cases = [
    (280, 620, 380),  # All above target
    (300, 350, 180),  # Only latency above
    (120, 500, 400),  # Cost + carbon above
]

print("Trained model predictions:")
for lat, cst, crb in test_cases:
    msgs = make_prompt(lat, cst, crb)
    result = pipe(msgs, return_full_text=False)
    output = result[0]["generated_text"].strip()
    print(f"  State: lat={lat}, cost={cst}, carbon={crb} → {output}")

## 8. Push to HuggingFace Hub (Optional)

In [ ]:
# Uncomment and run to push to HuggingFace Hub
# from huggingface_hub import login, whoami
# login(token="YOUR_HF_WRITE_TOKEN")
# username = whoami()["name"]
# model.push_to_hub(f"{username}/ecocloud-grpo-qwen")
# tokenizer.push_to_hub(f"{username}/ecocloud-grpo-qwen")
# print(f"✅ Model pushed to https://huggingface.co/{username}/ecocloud-grpo-qwen")